In [2]:
"""
Step 3: NG-RC style polynomial classifier on the dual-filter features (float64 reference).

Needs step2_filters.npz (written by step1_2_filters_baselines.py) in the working folder.
Feature vector per shot:  a = I_MF, b = Q_MF, c = I_RMF, d = Q_RMF  (each a sum over time bins)

Classifiers compared (all trained by ridge regression on targets -1/+1):
  lin2       : [a, b]                           -> same information as the plain matched filter
  lin4       : [a, b, c, d]                     -> adds the relaxation filter
  spec quad  : lin4 + [a^2, b^2, ab, ac, bd]    -> the 5 monomials listed in your spec
  full quad  : lin4 + all 10 pairwise products  -> checks if the spec's choice leaves anything out
Also a headroom probe: linear ridge on all 1540 raw values.
"""
import os
import numpy as np

# ------------------------------------------------------------------ config
DATA_DIR = r"D:\MSU\Qubit_readout_FPGA\python"
FILTER_FILE = "step2_filters.npz"
ALPHA_KEY = "0.05"            # which EMA alpha's filters to use: "None", "0.2", "0.05", "0.01"
T_EVAL = [385, 500, 578, 770]  # accumulation lengths (time bins) to compare
KINDS = ["lin2", "lin4", "spec quad", "full quad"]
CH = 20_000
RIDGE_LAMBDA = 1e-3
RAW_LAMBDA_REL = 1e-3          # ridge strength for the raw-trace probe (relative to N)
RUN_RAW_PROBE = True

Ts = np.array(T_EVAL)


def p(name):
    return os.path.join(DATA_DIR, name)


# ------------------------------------------------------------------ load
if not os.path.exists(FILTER_FILE):
    raise FileNotFoundError(f"{FILTER_FILE} not found. Run step1_2_filters_baselines.py to the end first.")
z = np.load(FILTER_FILE)
clf_idx = z["clf_idx"]
names = ("Wmf_I", "Wmf_Q", "Wrmf_I", "Wrmf_Q")
W = [z[f"{k}_a{ALPHA_KEY}"].astype(np.float32) for k in names]

Xtr = np.load(p("0528_X_train_0_770.npy"), mmap_mode="r")
Xte_m = np.load(p("0528_X_test_0_770.npy"), mmap_mode="r")
ytr = np.load(p("0528_y_train_0_770.npy")).astype(int)
yte = np.load(p("0528_y_test_0_770.npy")).astype(int)


def load_rows(X, idx):
    out = np.empty((len(idx), X.shape[1]), np.float32)
    for i in range(0, len(idx), CH):
        out[i:i + CH] = X[idx[i:i + CH]]
    return out


print("loading CLF and TEST sets ...")
Xclf = load_rows(Xtr, clf_idx)
yclf = ytr[clf_idx]
Xte = load_rows(Xte_m, np.arange(len(yte)))
print(f"CLF={len(yclf)}  TEST={len(yte)}  filters: EMA alpha={ALPHA_KEY}")


# ------------------------------------------------------------------ features
def cum_features(X):
    """(N, len(Ts), 4): the four filter sums, accumulated only up to each T in Ts."""
    N = len(X)
    out = np.empty((N, len(Ts), 4), np.float64)
    for i in range(0, N, CH):
        x = X[i:i + CH]
        I, Q = x[:, 0::2], x[:, 1::2]
        parts = [I * W[0], Q * W[1], I * W[2], Q * W[3]]
        for k, pk in enumerate(parts):
            out[i:i + CH, :, k] = np.cumsum(pk, axis=1, dtype=np.float64)[:, Ts - 1]
    return out


Fc_all = cum_features(Xclf)
Ft_all = cum_features(Xte)


def build(Z, kind):
    """Z = standardized base features (N,4). Returns the feature matrix for the chosen classifier."""
    a, b, c, d = Z.T
    if kind == "lin2":
        return Z[:, :2]
    if kind == "lin4":
        return Z
    if kind == "spec quad":
        extra = [a * a, b * b, a * b, a * c, b * d]
    elif kind == "full quad":
        extra = [Z[:, i] * Z[:, j] for i in range(4) for j in range(i, 4)]
    else:
        raise ValueError(kind)
    return np.c_[Z, np.stack(extra, 1)]


# ------------------------------------------------------------------ helpers
def ridge_fit(F, y, lam=RIDGE_LAMBDA):
    mu, sd = F.mean(0), F.std(0) + 1e-12
    Z = np.c_[np.ones(len(F)), (F - mu) / sd]
    t = 2.0 * y - 1.0
    w = np.linalg.solve(Z.T @ Z + lam * np.eye(Z.shape[1]), Z.T @ t)
    return mu, sd, w


def ridge_score(F, m):
    mu, sd, w = m
    return np.c_[np.ones(len(F)), (F - mu) / sd] @ w


def best_threshold(s, y):
    order = np.argsort(s)
    ss, ys = s[order], y[order]
    n0 = (ys == 0).sum()
    c1 = np.r_[0, np.cumsum(ys == 1)]
    c0 = np.r_[0, np.cumsum(ys == 0)]
    k = int(np.argmin(c1 + (n0 - c0)))
    if k == 0:
        return ss[0] - 1.0
    if k == len(ss):
        return ss[-1]
    return 0.5 * (ss[k - 1] + ss[k])


def fid(pred, y):
    f0 = np.mean(~pred[y == 0])
    f1 = np.mean(pred[y == 1])
    return f0, f1, 0.5 * (f0 + f1)


def paired_diff(correct_a, correct_b):
    """Mean difference in per-shot correctness (a - b) and its standard error."""
    d = correct_a.astype(float) - correct_b.astype(float)
    return d.mean(), d.std() / np.sqrt(len(d))


# ------------------------------------------------------------------ main comparison
res = {}
for j, T in enumerate(Ts):
    mu_b, sd_b = Fc_all[:, j, :].mean(0), Fc_all[:, j, :].std(0) + 1e-12
    Zc = (Fc_all[:, j, :] - mu_b) / sd_b
    Zt = (Ft_all[:, j, :] - mu_b) / sd_b
    for kind in KINDS:
        Fc, Ft = build(Zc, kind), build(Zt, kind)
        m = ridge_fit(Fc, yclf)
        s_c, s_t = ridge_score(Fc, m), ridge_score(Ft, m)
        thr = best_threshold(s_c, yclf)
        pred_t = s_t > thr
        f0, f1, F = fid(pred_t, yte)
        res[(kind, j)] = dict(
            F=F, f0=f0, f1=f1, F_clf=fid(s_c > thr, yclf)[2],
            correct=(pred_t == (yte == 1)), n_feat=Fc.shape[1],
            model=dict(mu_b=mu_b, sd_b=sd_b, mu=m[0], sd=m[1], w=m[2], thr=thr),
        )

print("\n=== Assignment fidelity on full test set (EMA alpha=%s) ===" % ALPHA_KEY)
print(f"{'classifier':>12s} {'#feat':>6s} | " + " ".join(f"T={t:<5d}" for t in Ts))
for kind in KINDS:
    row = " ".join(f"{res[(kind, j)]['F']:.4f} " for j in range(len(Ts)))
    print(f"{kind:>12s} {res[(kind, 0)]['n_feat']:6d} | {row}")

print("\n=== Gain over lin4, in fidelity points (+/- 1 standard error) ===")
for kind in ("spec quad", "full quad"):
    cells = []
    for j in range(len(Ts)):
        d, se = paired_diff(res[(kind, j)]["correct"], res[("lin4", j)]["correct"])
        cells.append(f"{100 * d:+.2f}+/-{100 * se:.2f}")
    print(f"{kind:>12s}        | " + "  ".join(cells))
print("\n=== Gain of lin4 over lin2 (value of the relaxation filter) ===")
cells = []
for j in range(len(Ts)):
    d, se = paired_diff(res[("lin4", j)]["correct"], res[("lin2", j)]["correct"])
    cells.append(f"{100 * d:+.2f}+/-{100 * se:.2f}")
print(f"{'lin4 - lin2':>12s}        | " + "  ".join(cells))
print("(A gain is only meaningful if it is clearly larger than its +/- value.)")

# best config chosen on the CLF set, not on test
best = max(res, key=lambda k: res[k]["F_clf"])
b = res[best]
print(f"\nBest config chosen on CLF set: '{best[0]}' with T={Ts[best[1]]}  "
      f"-> test F0={b['f0']:.4f} F1={b['f1']:.4f} F={b['F']:.4f}")
mdl = b["model"]
np.savez("step3_best_model.npz", kind=best[0], T=Ts[best[1]], alpha=ALPHA_KEY, **mdl)
print("saved step3_best_model.npz")


# ------------------------------------------------------------------ headroom probe: linear ridge on all raw values
if RUN_RAW_PROBE:
    print("\nraw-trace probe (linear ridge on all 1540 values) ...")
    D = Xclf.shape[1]
    mu = Xclf.mean(axis=0, dtype=np.float64)
    var = np.zeros(D)
    for i in range(0, len(Xclf), CH):
        var += ((Xclf[i:i + CH].astype(np.float64) - mu) ** 2).sum(0)
    sd = np.sqrt(var / len(Xclf)) + 1e-6
    t = 2.0 * yclf - 1.0
    t = t - t.mean()
    G = np.zeros((D, D))
    bvec = np.zeros(D)
    for i in range(0, len(Xclf), CH):
        Zr = (Xclf[i:i + CH].astype(np.float64) - mu) / sd
        G += Zr.T @ Zr
        bvec += Zr.T @ t[i:i + CH]
    w_raw = np.linalg.solve(G + RAW_LAMBDA_REL * len(Xclf) * np.eye(D), bvec)

    def raw_score(X):
        out = np.empty(len(X))
        for i in range(0, len(X), CH):
            out[i:i + CH] = ((X[i:i + CH].astype(np.float64) - mu) / sd) @ w_raw
        return out

    s_c, s_t = raw_score(Xclf), raw_score(Xte)
    thr = best_threshold(s_c, yclf)
    f0, f1, F = fid(s_t > thr, yte)
    print(f"Raw-trace linear ridge (1540 weights): F0={f0:.4f} F1={f1:.4f} F={F:.4f}")
    print("This is a rough ceiling for any LINEAR method on this data; compare with lin4 above.")

loading CLF and TEST sets ...
CLF=100000  TEST=100000  filters: EMA alpha=0.05

=== Assignment fidelity on full test set (EMA alpha=0.05) ===
  classifier  #feat | T=385   T=500   T=578   T=770  
        lin2      2 | 0.9561  0.9603  0.9594  0.9566 
        lin4      4 | 0.9564  0.9610  0.9608  0.9599 
   spec quad      9 | 0.9562  0.9610  0.9608  0.9599 
   full quad     14 | 0.9563  0.9614  0.9617  0.9617 

=== Gain over lin4, in fidelity points (+/- 1 standard error) ===
   spec quad        | -0.02+/-0.01  +0.00+/-0.01  +0.00+/-0.01  +0.00+/-0.01
   full quad        | -0.01+/-0.01  +0.04+/-0.01  +0.10+/-0.01  +0.18+/-0.02

=== Gain of lin4 over lin2 (value of the relaxation filter) ===
 lin4 - lin2        | +0.03+/-0.01  +0.07+/-0.01  +0.14+/-0.02  +0.33+/-0.02
(A gain is only meaningful if it is clearly larger than its +/- value.)

Best config chosen on CLF set: 'full quad' with T=770  -> test F0=0.9764 F1=0.9471 F=0.9617
saved step3_best_model.npz

raw-trace probe (linear ridge on